# Exoplanet Transit: Data Analysis

This notebook helps you load and analyse the light-sensor data from your model-exoplanet experiment.

It works with data exported from **phyphox**, **Physics Toolbox**, the **Arduino Science Journal**, or **any plain CSV** with a time column and a light/brightness column.

**You will produce:**
1. a **light curve** (light vs. time),
2. a **transit depth** value,
3. a plot showing **how the transit depth depends on the planet size**, and a test of which relationship fits your data best.

Work through the cells from top to bottom. **Above every cell there is a short note** telling you whether to simply run it, or to change something in it first.


## How to use this notebook

New to Google Colab? A few basics:

- **Run a cell** by clicking it and pressing **Shift + Enter** (or click the **▶** button on its left).
- **Run the cells in order, from top to bottom.** A cell often needs things set up by the cells above it, so if you skip one you may get an error (for example a `NameError`).
- **Comments start with a `#`.** These lines are notes for you, not instructions for the computer: Python ignores them, and the editor usually shows them in a different colour (green in Colab's default theme). They explain what the code does, or point out a line you may need to change.
- The first time you run something, Colab may warn that the notebook **was not authored by Google**. Click **Run anyway**; this notebook only does calculations and makes plots.
- If a cell shows a **red error**, read the **last line**, fix it (or ask your AI tutor) and run the cell again.
- Anything you type or produce here is temporary, which is fine for this lesson.


## 1. Upload your data

First, in your phone app, **export your recording as a CSV** and send it to your computer:
- **phyphox (Android only):** after your recording, tap the **three-dot menu (⋮) → Export Data**, choose **CSV (Comma, decimal point)** and send the file to yourself. phyphox packs the data in a **`.zip`** file: **upload the `.zip` as it is**, the notebook finds the data inside. (Other CSV options, such as a decimal comma, also work.)
- **Physics Toolbox:** stop the recording, tap the share/export button, choose CSV.
- **Arduino Science Journal (iPhone/iPad):** open the experiment, export it, and choose CSV.
- **Arduino Science Journal (Android):** open the experiment, **tap the recording**, then the **three-dot menu (⋮) → Share → turn on "Relative time" → save/send the file**. This gives a `.csv`.
  - ⚠️ If you instead share the *whole experiment* you get a **`.sj`** file (a Science Journal experiment archive). This notebook cannot read a `.sj` directly: use the per-recording CSV above. (If you do upload a `.sj` by mistake, the loading cell will remind you how to get the CSV.)

**▶ Run the cell below.** A **file picker will pop up**: choose your exported file. You do not need to change any code.

*(The cell also works outside Colab: if you ever run the notebook on your own computer, the picker will not appear, and instead it looks for a file named `your_data.csv` next to the notebook.)*

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()          # a file picker pops up: choose your CSV
    filename = next(iter(uploaded))
except Exception:
    # This runs ONLY if you are not in Colab (e.g. on your own computer):
    # it looks for a file called your_data.csv next to the notebook instead.
    uploaded = {}
    filename = 'your_data.csv'
print('Using file:', filename)

## 2. Load and preview the data

**▶ Run the cell below.** It reads your CSV and prints the **names of the columns** it found (`Columns found: ...`), then shows the first few rows so you can check the file loaded correctly. You will use those column names in the next cell. Nothing to change here.


In [ ]:
import io, re, zipfile
import numpy as np
import pandas as pd

def _read_table(raw_text):
    # Works out how the table is written: values separated by commas, semicolons
    # or tabs, and numbers with a decimal point (3.14) or a decimal comma (3,14).
    lines = [l for l in raw_text.splitlines() if l.strip()]
    header = lines[0] if lines else ''
    sample = '\n'.join(lines[1:6])
    sep = '\t' if '\t' in header else (';' if ';' in header else ',')
    decimal = ',' if (sep != ',' and re.search(r'\d,\d', sample)) else '.'
    try:
        d = pd.read_csv(io.StringIO(raw_text), sep=sep, decimal=decimal, engine='python')
        if d.shape[1] >= 2:
            return d
    except Exception:
        pass
    for s in [',', ';', '\t']:              # fallback: just try each separator
        try:
            d = pd.read_csv(io.StringIO(raw_text), sep=s, engine='python')
            if d.shape[1] >= 2:
                return d
        except Exception:
            pass
    return pd.read_csv(io.StringIO(raw_text))

def _text(raw_bytes):
    return raw_bytes.decode('utf-8-sig', errors='replace')   # utf-8-sig drops a hidden BOM

def load_data(name):
    # Get the raw bytes of the file (from the Colab upload, or from disk).
    if name in uploaded:
        data = uploaded[name]
    else:
        with open(name, 'rb') as f:
            data = f.read()
    low = name.lower()

    # An Excel file (for example phyphox's "Excel" export option).
    if low.endswith(('.xls', '.xlsx')):
        try:
            return pd.read_excel(io.BytesIO(data))
        except Exception:
            raise ValueError("Could not read this Excel file. Export your data as CSV "
                             "instead (in phyphox: Export Data -> CSV (Comma, decimal point)).")

    # A ZIP file (it starts with the bytes "PK"). phyphox exports its data as a
    # ZIP with the measurements in a CSV and extra information in a "meta" folder.
    # A Science Journal ".sj" experiment file is also a ZIP.
    if low.endswith(('.sj', '.zip')) or data[:2] == b'PK':
        try:
            z = zipfile.ZipFile(io.BytesIO(data))
            csvs = [n for n in z.namelist()
                    if n.lower().endswith('.csv') and 'meta/' not in n.lower()]
            # Prefer a file that sounds like light data, otherwise take the first one.
            csvs.sort(key=lambda n: 0 if re.search(r'light|illumin|lux', n.lower()) else 1)
            if csvs:
                print('Reading', csvs[0], 'from inside the zip file')
                return _read_table(_text(z.read(csvs[0])))
        except zipfile.BadZipFile:
            pass
        if low.endswith('.sj'):
            raise ValueError(
                "This is a Science Journal experiment file (.sj), which this notebook "
                "cannot read. On Android, export the RECORDING as CSV instead: open the "
                "experiment, tap the recording, then the three-dot menu (More) -> Share "
                "-> turn 'Relative time' ON -> save the .csv. Then run the upload cell "
                "again and pick that .csv file.")
        raise ValueError("This zip file contains no CSV table with measurements. "
                         "Export your recording as CSV and upload that file.")

    # Otherwise it's a CSV / plain-text table.
    return _read_table(_text(data))

df = load_data(filename)
df.columns = [str(c).strip() for c in df.columns]
print('Columns found:', list(df.columns))     # <-- you will use these names below
df.head()

Next, the notebook picks out which column is the **time** and which is the **light** measurement, automatically.

**▶ Run the cell below**, then read the two lines it prints (`Time column:` and `Light column:`).

- If both are **correct**, do nothing and carry on to the next cell.
- If either is **wrong**, fix it *in this cell*:
    1. Look at the `Columns found:` list printed by the cell above.
    2. Change `override_time_col = None` into the correct name **in quotes**, for example `override_time_col = 'time'`.
    3. Do the same with `override_light_col` if the light column is wrong, e.g. `override_light_col = 'Illuminance (lx)'`.
    4. **Run the cell again** and check the two printed lines are now correct.

    Leave a line as `None` if that column was already guessed correctly.


In [ ]:
def pick_column(df, keywords):
    # Looks for a column whose name contains one of these words.
    for c in df.columns:
        if any(k in c.lower() for k in keywords):
            return c
    return None

time_col  = pick_column(df, ['time', 'timestamp', 'seconds', 'elapsed'])
light_col = pick_column(df, ['illuminance', 'lux', 'lx', 'light', 'bright', 'intensity', 'value'])

# If the automatic guess above failed, fall back to sensible defaults:
if time_col is None:
    time_col = df.columns[0]                 # use the first column as the time
if light_col is None:
    for c in df.columns:                     # use the first column that contains numbers
        if c != time_col and pd.to_numeric(df[c], errors='coerce').notna().mean() > 0.5:
            light_col = c
            break

# ---- CHANGE THESE TWO LINES ONLY IF the columns printed below are wrong. ----
# Copy the correct name from the "Columns found:" list and put it in quotes.
# Leave a line as None if that column was already guessed correctly.
override_time_col  = None      # example:  override_time_col  = 'time'
override_light_col = None      # example:  override_light_col = 'Illuminance (lx)'
# ---------------------------------------------------------------------------
if override_time_col:
    time_col = override_time_col
if override_light_col:
    light_col = override_light_col

print('Time column :', time_col)
print('Light column:', light_col)
assert time_col and light_col, 'Could not find the columns. Set them by hand in the two lines above.'

**▶ Run the cell below.** It turns the two columns into numbers and measures the time in seconds from the start of the recording, then prints how many data points you have. Nothing to change.


In [ ]:
# Turn the two columns into numbers, and measure time in seconds from the start.
t = pd.to_numeric(df[time_col], errors='coerce')
if t.isna().mean() > 0.5:                 # the time column looks like a clock time, not a number
    tt = pd.to_datetime(df[time_col], errors='coerce', format='mixed')
    t = (tt - tt.min()).dt.total_seconds()
light = pd.to_numeric(df[light_col], errors='coerce')

ok = t.notna() & light.notna()            # keep only rows where both values are real numbers
time_s = (t[ok] - t[ok].min()).to_numpy()
light  = light[ok].to_numpy()

# Some apps (e.g. Arduino Science Journal) record time in MILLISECONDS. This
# experiment only lasts seconds, so if the recording looks far too long, the
# time must be in milliseconds -> convert to seconds.
if time_s.size > 1 and time_s.max() > 600:    # > 10 minutes is implausible here
    time_s = time_s / 1000.0
    print('(time looked like milliseconds, converted to seconds)')

print(f'{len(time_s)} data points over {time_s.max():.1f} s')

## 3. Plot the light curve  *(deliverable 1)*

This is your transit light curve: brightness versus time. You should see a **dip** where the model planet passed in front of the star.

**▶ Run the cell below** to draw and save it. This figure is your first deliverable; nothing to change.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(time_s, light, lw=1)
plt.xlabel('Time (s)')
plt.ylabel(f'Light  ({light_col})')
plt.title('Transit light curve')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('light_curve.png', dpi=150)
plt.show()

## 4. Measure the transit depth

The **transit depth** is how much the light dropped during the transit:

$$\text{depth}\,(\%) = \frac{\text{baseline} - \text{minimum}}{\text{baseline}} \times 100$$

- **baseline** = the normal brightness outside the dip,
- **minimum** = the lowest brightness during the dip.

**▶ Run the cell below** to measure the depth and mark the baseline (green) and minimum (red) on your light curve. Usually there is nothing to change. **Only if** your recording is noisy or has more than one dip: set `t_start` and `t_end` (the start and end time of the dip, in seconds) at the top of the cell, then run it again.


In [ ]:
# The transit depth is measured automatically from the whole recording.
# CHANGE this only if your recording is noisy or has more than one dip: set the
# start and end time (in seconds) of the dip. Otherwise leave them as None.
t_start, t_end = None, None          # example:  t_start, t_end = 5.0, 9.0

if t_start is not None and t_end is not None:
    in_transit = (time_s >= t_start) & (time_s <= t_end)
    baseline = float(np.median(light[~in_transit]))
    minimum  = float(np.min(light[in_transit]))
else:
    baseline = float(np.median(light))
    minimum  = float(np.min(light))

depth_pct = (baseline - minimum) / baseline * 100
print(f'baseline = {baseline:.2f},  minimum = {minimum:.2f},  depth = {depth_pct:.2f}%')

# Show the baseline and minimum on the light curve.
plt.figure(figsize=(8, 4))
plt.plot(time_s, light, lw=1)
plt.axhline(baseline, color='green', ls='--', label='baseline')
plt.axhline(minimum, color='red', ls='--', label='minimum')
plt.xlabel('Time (s)'); plt.ylabel(f'Light ({light_col})')
plt.title(f'Transit depth = {depth_pct:.2f}%')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 5. Compare planet sizes  *(deliverable 2)*

Run steps 1 to 4 again for **each planet size** you tested and note the transit depth each time.

**✎ Edit the cell below, then run it.** Type your `(size, depth)` pairs into the list: replace the numbers in the first row with your own, and add one row per planet size you tested (copy the row and change the numbers). You need **at least three** rows.

*Use one consistent measure of size for all planets, for example the diameter in cm.*

Running the cell plots **transit depth against planet size**. Look carefully at the result:
- Do the points lie on a **straight line**, or do they **curve**?
- What would a straight line mean? And what does it mean if the line curves upwards?

In [ ]:
# Type one (size, depth) pair per planet size you tested, one per line.
# Replace the numbers in the first row with your own, then add a row for each
# extra size (copy the row and change the numbers). Keep the ( ) and the comma.
results = [
    (1.0, 4.2),      # <-- replace 1.0 and 4.2 with YOUR size and depth
    # (1.5, 9.1),    # <-- copy this line for each extra size (and remove the #)
]

size  = np.array([r[0] for r in results], float)
depth = np.array([r[1] for r in results], float)

if len(results) >= 2:
    plt.figure(figsize=(7, 4))
    plt.scatter(size, depth, zorder=3)
    plt.xlabel('Planet size'); plt.ylabel('Transit depth (%)')
    plt.title('Transit depth against planet size')
    plt.xlim(left=0); plt.ylim(bottom=0)
    plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig('depth_vs_size.png', dpi=150)
    plt.show()
else:
    print('Add at least two (size, depth) results above, then run this cell again.')

## 6. Test a model: which relationship fits?

Scientists often test an idea by asking: *if my model is right, which graph should give a **straight line**?* A straight line through the origin (0, 0) means that the transit depth is **proportional** to what you put on the x-axis.

Think first: why should the line go through (0, 0)? What does a planet of size 0 do to the light?

**✎ Edit the cell below, then run it.** The cell plots the transit depth against `x` and draws the best straight line through (0, 0). Start with `x = size`. Then **try other options** by changing that one line, for example:
- `x = size`
- `x = size**2` &nbsp; (size squared, `size × size`)
- `x = size**3` &nbsp; (size cubed)
- or your own idea, if you have a reason for it.

Run the cell after each change and compare the graphs. **Which choice puts your points closest to a straight line through (0, 0)?** Can you explain *why* that one fits, using what blocks the light? Keep the figure of your best choice: it is your result plot.

In [ ]:
# CHANGE THIS LINE to test a different model, then run the cell again.
x = size                       # try for example:  x = size**2   or   x = size**3
x_label = 'size'               # describe your choice for the axis, e.g. 'size squared'

if len(results) >= 2:
    # Best straight line through the origin: depth = slope * x
    slope = float(np.sum(x * depth) / np.sum(x * x))
    predicted = slope * x
    spread = float(np.sqrt(np.mean((depth - predicted) ** 2)))   # how far the points are from the line

    xx = np.linspace(0, x.max() * 1.1, 50)
    plt.figure(figsize=(7, 4))
    plt.scatter(x, depth, zorder=3, label='your measurements')
    plt.plot(xx, slope * xx, '--', color='gray', label='straight line through (0, 0)')
    plt.xlabel(f'Planet {x_label}'); plt.ylabel('Transit depth (%)')
    plt.title(f'Transit depth against planet {x_label}')
    plt.xlim(left=0); plt.ylim(bottom=0)
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig('depth_vs_model.png', dpi=150)
    plt.show()
    print(f'Average distance of your points from the line: {spread:.2f} percentage points.')
    print('The smaller this number, the better this straight line fits your data.')
else:
    print('Add at least two (size, depth) results in the cell above first.')

## 7. Download your figures

**▶ Run the cell below** to download your figures (`light_curve.png`, `depth_vs_size.png` and `depth_vs_model.png`, which holds the model you tested last) so you can add them to your report. Nothing to change.

In [ ]:
try:
    from google.colab import files
    for f in ['light_curve.png', 'depth_vs_size.png', 'depth_vs_model.png']:
        try:
            files.download(f)
        except Exception as e:
            print('Could not download', f, '-', e)
except Exception:
    print('Not in Colab - the figures are saved next to this notebook.')

---
**Need to change the code?** Ask **Emmy** (or your own AI tutor): describe what you want to do and paste any error messages. Always check that the output uses *your* data and that the axes and units are labelled correctly: the AI can be wrong.
